### Time Series

Data indexed by time, where observations are NOT independent, a direct violation of the IID assumption every other model in this series relies on (see `classical-ml.ipynb`'s Linear Regression "independence of errors" assumption, this is exactly the case where it's violated on purpose, by construction). Covers decomposition, autocorrelation, stationarity, exponential smoothing, ARIMA.

#### 0. Decomposition: trend + seasonality + residual

Additive model: y = trend + seasonal + residual. Worked example, CONSTRUCTED from known pieces (to show how they combine, real decomposition works backward from raw y to recover these pieces, done in code below):
```
trend (linear, +1 per step):     [10, 11, 12, 13, 14, 15, 16, 17]
seasonal (period 4, repeating):  [+2, +3,  0, -3, +2, +3,  0, -3]
residual:                        [ 0,  0,  0,  0,  0,  0,  0,  0]  (clean construction, no noise)

y = trend + seasonal:            [12, 14, 12, 10, 16, 18, 16, 14]
```
Notice the seasonal pattern repeats identically every 4 steps (+2,+3,0,-3), while the trend keeps climbing underneath it, the observed series y is their sum. Real decomposition (statsmodels' `seasonal_decompose`, used in code below) estimates trend via a moving average over one full season's width, then estimates the seasonal component from what's left after removing trend, then whatever remains is the residual.

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose

trend = np.arange(10, 18)
seasonal = np.tile([2, 3, 0, -3], 2)
y = trend + seasonal

series = pd.Series(y, index=pd.date_range("2026-01-01", periods=8, freq="D"))
result = seasonal_decompose(series, model="additive", period=4, extrapolate_trend="freq")

print("recovered trend:\n", result.trend.values)
print("recovered seasonal:\n", result.seasonal.values)

#### 1. Autocorrelation (ACF), worked by hand

Correlation of a series with a LAGGED version of itself, does knowing y_t tell you something about y_(t+1) (or further out). Formula (lag-1): ACF(1) = sum(dev_t * dev_(t+1)) / sum(dev_t^2), where dev_t = y_t - mean(y).

Worked example, series [1, 3, 2, 4, 3, 5], mean=3:
```
deviations: [-2, 0, -1, 1, 0, 2]

lag-1 products: (-2*0), (0*-1), (-1*1), (1*0), (0*2) = 0, 0, -1, 0, 0
sum of products = -1
sum of squared deviations = 4+0+1+1+0+4 = 10

ACF(1) = -1/10 = -0.1
```
Weak negative autocorrelation at lag 1, knowing today's value gives almost no useful signal about tomorrow's in this particular toy series. A strongly autocorrelated series (ACF near +1 or -1 at some lag) is exactly what AR models below exploit, if y_t reliably predicts y_(t+1), a model can be built directly on that relationship.

In [ ]:
from statsmodels.tsa.stattools import acf

toy_series = np.array([1, 3, 2, 4, 3, 5])
acf_values = acf(toy_series, nlags=2, fft=False)
print("ACF at lags 0,1,2:", acf_values.round(3))

#### 2. Stationarity and differencing, worked by hand

Stationary: mean and variance stay roughly constant over time, no trend, no systematically growing/shrinking spread. Most classical time series models (ARIMA below) assume or require stationarity to work correctly, a series with an obvious trend has a mean that keeps changing, violating that assumption outright.

Differencing removes a trend: y'_t = y_t - y_(t-1). Worked example, a perfectly linear trend, [10, 13, 16, 19, 22] (each step +3):
```
differenced: [13-10, 16-13, 19-16, 22-19] = [3, 3, 3, 3]
```
The differenced series is now CONSTANT, mean and variance no longer change over time, stationary by construction. This is exactly what the "I" (integrated) in ARIMA refers to, how many times you need to difference the raw series before it becomes stationary enough for the AR/MA parts to work correctly.

In [ ]:
linear_trend_series = np.array([10, 13, 16, 19, 22])
differenced = np.diff(linear_trend_series)
print("differenced series:", differenced, "(constant -> stationary)")

#### 3. Exponential smoothing, worked by hand

Formula: S_t = alpha*y_t + (1-alpha)*S_(t-1), a weighted average favoring recent observations, but never fully discarding older ones (they're baked into S_(t-1)).

Worked example, series [10, 12, 11, 15], alpha=0.3, S_0=y_0=10:
```
S_1 = 0.3*12 + 0.7*10 = 3.6 + 7.0 = 10.6
S_2 = 0.3*11 + 0.7*10.6 = 3.3 + 7.42 = 10.72
S_3 = 0.3*15 + 0.7*10.72 = 4.5 + 7.504 = 12.004
```
The smoothed series [10, 10.6, 10.72, 12.0] lags behind and dampens the raw series' [10, 12, 11, 15] swings, a simple, cheap forecasting baseline, the next forecast is just the latest S_t carried forward.

In [ ]:
from statsmodels.tsa.holtwinters import SimpleExpSmoothing

series_smooth = np.array([10, 12, 11, 15], dtype=float)
model = SimpleExpSmoothing(series_smooth).fit(smoothing_level=0.3, optimized=False)
print("fitted (smoothed) values:", model.fittedvalues.round(3))

#### 4. ARIMA

AR(p), autoregressive: predict y_t from its own past p values. AR(1): y_t = c + phi*y_(t-1) + error_t, phi is the same kind of learned coefficient as a linear regression weight (see `classical-ml.ipynb`), just applied to the series' own lagged values as the "feature."

I(d), integrated: how many times to DIFFERENCE the series (section 2 above) before fitting, to make it stationary.

MA(q), moving average: predict y_t from the past q FORECAST ERRORS (not past values), y_t = c + theta*error_(t-1) + error_t, models the idea that a recent surprise (large forecast error) tends to have some lingering effect on the next observation.

ARIMA(p,d,q) combines all three: difference d times for stationarity, then fit an AR(p) + MA(q) model on the differenced series. Choosing p, d, q is typically done by examining ACF/PACF (partial autocorrelation) plots or via automated search (grid search over small p,d,q combinations, similar in spirit to `hyperparameter-tuning.ipynb`'s grid search, minimizing an information criterion like AIC instead of a validation score).

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

rng = np.random.default_rng(0)
arima_series = np.cumsum(rng.normal(size=50)) + 50  # a random-walk-with-drift-like series

model = ARIMA(arima_series, order=(1, 1, 1))  # p=1, d=1, q=1
fitted = model.fit()
print(fitted.summary().tables[1])

forecast = fitted.forecast(steps=3)
print("\n3-step forecast:", forecast)